In [1]:
# Save total mortality by country for all ensemble members

In [1]:
import xarray as xr
import numpy as np

In [2]:
# === Latitude weighting sum ===
def weighted_sum(da):
    weights = np.cos(np.deg2rad(da.lat))
    weights.name = "weights"
    new_da = da.weighted(weights).sum(("lon", "lat"))
    return new_da

In [ ]:
# === Path config ===
MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"

country_mask = xr.open_dataarray(f"{MASK_DIR}GBD_Country_Masks_0.10_popgrid_newlabels.nc")

In [3]:
# === Path config ===
MORTALITY_DIR = "/glade/work/awells/air_quality/CESM/mortality/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    ensembles = []
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        da = xr.open_dataarray(f"{MORTALITY_DIR}Mortality_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")

        countries = []
        # Loop over countries and sum the mortality for each country
        for i in range(len(country_mask.country)):
            print(f"Country number {i}")
            mask = country_mask.isel(country=i)
            M_country = weighted_sum(xr.where(mask == 1, da.sel(quantile="mean"), np.nan)) # mean mortality of country (based on BMR mean)
            countries.append(M_country.drop_vars("country", errors='ignore'))
        mortality_country = xr.concat(countries, dim=xr.DataArray(country_mask["country"], dims="country", name="country"))

        ensembles.append(mortality_country)

    countries_ens = xr.concat(ensembles, dim=xr.DataArray(np.arange(1, len(ensembles)+1), dims="ensemble", name="ensemble"))
    print(f"Saving country mortality to {MORTALITY_DIR}")
    countries_ens.to_netcdf(f"{MORTALITY_DIR}Mortality_Country_sum_CESM2_{scenario}_{dates}.nc")

Processing SSP245, Ensemble 01
Country number 0
Country number 1
Country number 2
Country number 3
Country number 4
Country number 5
Country number 6
Country number 7
Country number 8
Country number 9
Country number 10
Country number 11
Country number 12
Country number 13
Country number 14
Country number 15
Country number 16
Country number 17
Country number 18
Country number 19
Country number 20
Country number 21
Country number 22
Country number 23
Country number 24
Country number 25
Country number 26
Country number 27
Country number 28
Country number 29
Country number 30
Country number 31
Country number 32
Country number 33
Country number 34
Country number 35
Country number 36
Country number 37
Country number 38
Country number 39
Country number 40
Country number 41
Country number 42
Country number 43
Country number 44
Country number 45
Country number 46
Country number 47
Country number 48
Country number 49
Country number 50
Country number 51
Country number 52
Country number 53
Country